## Definizione della classe circuit 
Si definisce la classe circuit in grado di rappresentare circuiti a n qubit e m time-steps come stati quantistici

In [ ]:
import numpy as np
import csv
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Operator

class Circuit:
    
    def __init__(self,shape,steps):  
        self._shape = shape
        
        # converts circuit to quantum state
        self._state = self._state_of_step(steps[0])
        for step in steps[1:]:
            self._state = np.kron(self._state, self._state_of_step(step))
            
    # returns quantum state from step list
    def _state_of_step(self,step):
        state = np.zeros((self._shape.get_n_steps()),dtype=np.float64)
        state[self._shape.get_step_index(step)] = 1.
        return state
    # return step from single step quantum state
    def _step_of_state(self,state):
        return self._shape._steps[np.where(state==1.)[0][0]]
    
    def get_n_qubits(self):
        return self._shape.get_n_qubits()
    def get_n_steps(self):
        return round(np.emath.logn(self._shape.get_n_steps(), self._state.shape[0]))
    
    # converts quantum state to human readable circuit
    def get_steps(self):
        states = [None for i in range(self.get_n_steps())]
        current_step = self.get_state()
        previous_step = np.zeros(( int(self._state.shape[0]/self._shape.get_n_steps()) ), dtype=np.float64)
        current_found = False
        for i in range(len(states)):
            for j in range(previous_step.shape[0]):
                block = current_step[j*self._shape.get_n_steps() : (j+1)*self._shape.get_n_steps()]
                if 1. in block:
                    previous_step[j] = 1.
                    if not current_found:
                        states[len(states)-1-i] = block.copy()
                        current_found = True
                else:
                    previous_step[j] = 0.
            current_step = previous_step.copy()
            previous_step = np.zeros(( int(current_step.shape[0]/self._shape.get_n_steps()) ), dtype=np.float64)
            current_found = False
            
        return [self._step_of_state(state) for state in states]
            
    def get_state(self):
        return self._state.copy()

    # saves the circuit in a csv file
    def to_csv(self,filename):
        with open(filename, 'w', newline='') as fout:
            writer = csv.writer(fout, delimiter=",")
            for step in self.get_steps():
                writer.writerow( step )
        
        
    #static method to create a circiut from a csv file
    def load_csv(filename):
        with open(filename, 'r', newline='') as fin:
            reader = csv.reader(fin, delimiter=",")
            steps = [row for row in reader]
            return Circuit(shape,steps)
    load_csv = load_csv
    
    # returns the corresponding qiskit.circuit.QuantumCircuit
    def to_qiskit(self):
        steps = self.get_steps()
        circuit = QuantumCircuit(self._shape.get_n_qubits())
        for step in steps:
            for i in range(len(step)):
                    if step[i][-1] == "c":
                        for j in range(i+1,len(step)):
                            if step[j][:-1] == step[i][:-1] and step[j][-1] == "t":
                                if step[i][:-1] == "S":
                                    circuit.swap(i,j)
                                elif step[i][:-1] == "CZ":
                                    circuit.cz(i,j)
        return circuit

    # draws the circuit using qiskit.circuit.QuantumCircuit.draw() method
    def draw(self,mpl=False):
        if not mpl:
            return self.to_qiskit().draw(output='text')
        else:
            return self.to_qiskit().draw("mpl")
        
    
    # returns true if c1 and c2 (Cricuit obejcts) implement the same algorithm
    def equivalent(c1,c2):
        U = Operator(c1.to_qiskit().compose(c2.to_qiskit().inverse())).to_matrix()
        return np.linalg.norm(U - np.identity( 2**c1.get_n_qubits() )) == 0
    equivalent = equivalent
        

con associata la classe shape che gestisce i possibili timesteps dato l'insieme dei gate disponibili ed il numero di qubit

In [ ]:
class CircuitShape:
    def __init__(self,n_qubits,gates):
        self._n_qubits = n_qubits
        self._steps = []
        self._find_steps(gates,[],0)
        
        
    def _find_steps(self,gates,step,length):
        if length == self._n_qubits:
            self._steps.append(step)
        else:
            for gate_length in gates.keys():
                for gate in gates[gate_length]:
                    if length+gate_length <= self._n_qubits:
                        new_step = step.copy()
                        for block in gate:
                            new_step.append(block)
                        self._find_steps(gates,new_step,length+gate_length)
    
    def get_n_qubits(self):
        return self._n_qubits

    def get_step_index(self,step):
        return self._steps.index(step)

    def get_n_steps(self):
        return len(self._steps)
    
    def print_steps(self,auto_print=True):
        out = ""
        for i in range(len(self._steps)):
            out += str(i) + ": " + str(self._steps[i]) + "\n"
        if auto_print: print(out)
        return out

Le due classi una volta scritte e testate vengono aggiunte al modulo QuantumCompiling

In [6]:
#from QuantumCompiling import Circuit, CircuitShape

test del calcolo dei possibili time-step con la classe shape

In [7]:
n_qubits = 4
gates = { 
    1 : [ ["ID"] ],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    # ,3 : [ ["Sc","ID","St"] ],
    # 4 : [ ["Sc","Sc","St","St"] ]
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); pass

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['Sc', 'St', 'ID', 'ID']
6: ['Sc', 'St', 'Sc', 'St']
7: ['Sc', 'St', 'CZc', 'CZt']
8: ['CZc', 'CZt', 'ID', 'ID']
9: ['CZc', 'CZt', 'Sc', 'St']
10: ['CZc', 'CZt', 'CZc', 'CZt']



Test della classe Circuit: calcolo dello stato dato un circuito human readable e operazione inversa

In [8]:
steps = [
    ["ID","ID","ID","ID"],
    ["Sc","St","ID","ID"],
    ["Sc","St","ID","ID"]
]

# steps3 = [
#     ["ID","ID","ID"],
#     ["Sc","St","ID"],
#     ["CZc","CZt","ID"],
#     ["ID","CZc","CZt"]
# ]

circuit = Circuit(shape,steps)


print("Stato:", circuit.get_state())
print("Conversione da stato a steps:",circuit.get_steps())
#circuit.draw()

Stato: [0. 0. 0. ... 0. 0. 0.]
Conversione da stato a steps: [['ID', 'ID', 'ID', 'ID'], ['Sc', 'St', 'ID', 'ID'], ['Sc', 'St', 'ID', 'ID']]


Test dei metodi di caricamento e salvataggio da/su csv

In [9]:
c1 = Circuit.load_csv("c1.csv",shape)
print("Stato:",c1.get_state())
print("Conversiona da stato a steps", c1.get_steps())

#salvataggio su csv
c1.to_csv("c1_out.csv")
print("Circuito salvato:", Circuit.load_csv("c1_out.csv",shape).get_steps())

Stato: [0. 0. 0. ... 0. 0. 0.]
Conversiona da stato a steps [['ID', 'ID', 'ID', 'ID'], ['Sc', 'St', 'ID', 'ID'], ['ID', 'Sc', 'St', 'ID'], ['ID', 'ID', 'CZc', 'CZt']]
Circuito salvato: [['ID', 'ID', 'ID', 'ID'], ['Sc', 'St', 'ID', 'ID'], ['ID', 'Sc', 'St', 'ID'], ['ID', 'ID', 'CZc', 'CZt']]


Test del metodo to qiskit

In [10]:
circuit.to_qiskit().draw()

q_0: ─X──X─
      │  │ 
q_1: ─X──X─
           
q_2: ──────
           
q_3: ──────

Si utilizzano i metodi qiskit per calcolare se due circuiti sono o non sono equivalenti.

In [12]:
c1 = Circuit.load_csv("c1.csv",shape)
c2 = Circuit.load_csv("c2.csv",shape)
c3 = Circuit.load_csv("c3.csv",shape)

print("Equivalenza c1-c2:",Circuit.equivalent(c1,c2))
print("Equivalenza c1-c3:",Circuit.equivalent(c1,c3))

print(c1.draw())
print(c2.draw())
print(c3.draw())

Equivalenza c1-c2: True
Equivalenza c1-c3: False
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─■───────
      │       
q_1: ─■──X──X─
         │  │ 
q_2: ────X──X─
              
q_3: ─────────
              
